In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re
import plotly.express as px
import plotly.graph_objects as go


### Загрузка данных

In [ ]:
df = pd.read_csv("NetflixShows.csv", sep=';')

In [ ]:
df_clean = df.drop_duplicates().copy()
df_clean['ratingLevel'] = df_clean['ratingLevel'].fillna('No description available')
df_clean = df_clean[df_clean['release year'] > 1950]
df_clean = df_clean.dropna()


print("\nПропущенные значения:")
print(df_clean.isnull().sum())
print(f"\nПроцент пропусков в 'user rating score': {df_clean['user rating score'].isnull().sum() / len(df_clean) * 100:.1f}%")

# **1. Титульник**

[Слайд 1]


Групповой проект №1 "Netflix"

Cовременные методы анализа данних и машинного обучения



Антонова Василиса, ББИ2409

Криворученко Кира, ББИ2409

Лавреньтев Роман, ББИ2409

Полякова Анастасия, ББИ2409

Шашуков Анатолий, ББИ2409

# **2. Описание исходных данных**

[слайд 2]

## Описание данных

In [ ]:
df.head()

title - название шоу

rating - рейтинг шоу. Например: G, PG, TV-14, TV-MA

ratingLevel - описание рейтинговой группы и особенностей шоу

ratingDescription - рейтинг шоу, закодированный числом

release year - год выпуска шоу

user rating score - оценка пользователей

user rating size - общий рейтинг пользователей

### БАЗОВАЯ СТАТИСТИКА:
Количество уникальных рейтинговых групп: 12

Рейтинговые группы: ['G', 'NR', 'PG', 'PG-13', 'R', 'TV-14', 'TV-G', 'TV-MA', 'TV-PG', 'TV-Y', 'TV-Y7', 'TV-Y7-FV']

Диапазон годов выпуска: 1978 - 2017

Средняя оценка пользователей: 81.5

Медианная оценка пользователей: 84.0

In [ ]:
print("БАЗОВАЯ СТАТИСТИКА:")
print(f"Количество уникальных рейтинговых групп: {df_clean['rating'].nunique()}")
print(f"Рейтинговые группы: {sorted(df_clean['rating'].unique())}")
print(f"\nДиапазон годов выпуска: {df_clean['release year'].min()} - {df_clean['release year'].max()}")
print(f"Средняя оценка пользователей: {df_clean['user rating score'].mean():.1f}")
print(f"Медианная оценка пользователей: {df_clean['user rating score'].median():.1f}")

[слайд 3]

## Первичный анализ данных

Типы признаков

In [ ]:
df.info()

In [ ]:
#df.describe()

# **3. Анализ дубликатов и ошибок**

[слайд 4]

## Проверка пропусков

In [ ]:
print("\nПропущенные значения:")
print(df.isnull().sum())
print(f"\nПроцент пропусков в 'user rating score': {df['user rating score'].isnull().sum() / len(df) * 100:.1f}%")

## Проверка дубликатов

In [ ]:
print(f"\nПолных дубликатов: {df.duplicated().sum()}")

## Анализ дубликатов ошибок


In [ ]:
df_for_dupl = df.copy()
df_for_dupl['is dublicated'] = df_for_dupl.duplicated()

duplicates_data = df_for_dupl.groupby('rating')['is dublicated'].sum()
total_counts = df_for_dupl.groupby('rating')['title'].count()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=duplicates_data.index,
    y=duplicates_data.values,
    name='Количество дубликатов',
    marker_color='black'
))
fig.add_trace(go.Bar(
    x=total_counts.index,
    y=total_counts.values,
    name='Количество продуктов',
    marker_color='red'
))

fig.update_layout(
    width=1800,
    height=600,
    xaxis_title='Рейтинги',
    yaxis_title='Количество',
    template="plotly_white"
)
fig.show()

# **4. Целевая переменная**

[слайд 5]

## Целевая переменная

User rating score

Цель проекта: выявить факторы, которые влияют на пользовательскую оценку контента (user rating score), и определить, какие характеристики шоу (возрастной рейтинг, тематика, уровень жёсткости, год выпуска и т.д.) связаны с более высоким восприятием аудитории.

# **5. Причесывание**

[слайд 6]

## Очистка данных

Удаление полных дубликатов, заполнение пустых данных в rating level, удаление строк с NA целевой переменной

In [ ]:
df_clean = df.drop_duplicates()
df_clean['ratingLevel'] = df_clean['ratingLevel'].fillna('No description available')
df_clean = df_clean[(df['release year'] > 1950)]
df_clean = df_clean.dropna()



print("\nПропущенные значения:")
print(df_clean.isnull().sum())
print(f"\nПроцент пропусков в 'user rating score': {df_clean['user rating score'].isnull().sum() / len(df_clean) * 100:.1f}%")

## Стандартизация названий рейтингов

In [ ]:
df_clean['rating'] = df_clean['rating'].str.strip()

In [ ]:
df_clean

# **6. Feature Engineering**

[слайд 7]

## Создание id для различия одинаковых названий

In [ ]:
df_clean['unique_id'] = (
    df_clean['title'] + '_' +
    df_clean['release year'].astype(str) + '_' +
    df_clean['rating']
)

## Добавление категориальных признаков



Возрастная группа, эпоха выпуска, качество рейтинга

In [ ]:
def categorize_age_group(rating):
    kids = ['TV-Y', 'TV-Y7', 'TV-Y7-FV', 'G', 'TV-G']
    family = ['PG', 'TV-PG']
    teens = ['PG-13', 'TV-14']
    mature = ['R', 'TV-MA', 'NC-17']
    unrated = ['NR', 'UR']

    if rating in kids:
        return 'Kids (0-6)'
    elif rating in family:
        return 'Family (7-12)'
    elif rating in teens:
        return 'Teens (13-17)'
    elif rating in mature:
        return 'Adults (18+)'
    else:
        return 'Unrated'

def categorize_era(year):
    if year < 2000:
        return '1970s-1990s'
    elif year < 2010:
        return '2000s'
    elif year < 2015:
        return '2010-2014'
    elif year < 2017:
        return '2015-2016'
    else:
        return '2017+'


df_clean['era'] = df_clean['release year'].apply(categorize_era)
df_clean['age_category'] = df_clean['rating'].apply(categorize_age_group)

In [ ]:
df_clean

# **7. Гипотезы + зависимости**

[слайд 8]

## Какие оценки пользователи ставят чаще всего

In [ ]:
import plotly.express as px

fig = px.histogram(
    df_clean,
    x="user rating score",
    nbins=60,
    histnorm="probability density",   # превращаем в плотность
    opacity=0.7,
    title="Распределение пользовательских оценок (KDE-аналог)"
)

# добавляем плавную линию плотности
fig.update_traces(marker_color="red")

fig.update_layout(
    xaxis_title="Оценки пользователей",
    yaxis_title="Плотность",
    template="plotly_white"
)

fig.show()


**Итог для бизнеса:**

1) Основная масса оценок сосредоточена в диапазоне от 77 до 95 баллов.
Это значит, что пользователи в целом склонны ставить высокие оценки — негативные или очень низкие отзывы встречаются редко. У Netflix устойчивый уровень доверия и удовлетворённости аудитории.
2) Распределение смещено вправо, что говориот о том пользователи чаще смотрят то, что им нравится, и реже оставляют оценки на слабые шоу. Так же это сивдетельсвуте о хорошей рекомендательной системе платфоры. Разница между “просто хорошим” и “отличным” контентом трудно увидеть по одной только метрике user rating score. Нужно дополнять user rating score метриками вовлечённости. При выборе направлений для инвестиций лучше смотреть на средние оценки внутри категорий, а не по всей платформе.



In [ ]:
df_clean_sicne1990 = df_clean[df_clean['release year'] >= 1995 ]

fig = px.density_contour(
    df_clean_sicne1990,
    x="release year",
    title="Плотность распределения годов выпуска контента",
    color_discrete_sequence=["crimson"]
)

fig.update_traces(contours_coloring="fill", contours_showlabels=True)
fig.update_layout(
    xaxis_title="Год выпуска",
    yaxis_title="Плотность",
    template="plotly_white",
    showlegend=False
)
fig.show()

## Оценка успешности проектов за 2016-2017

Количество релизов 2016: 86

Количество релизов 2017: 16

Средняя оценка 2016: 84.3

Средняя оценка 2017: 88.1

Вывод:

In [ ]:
df_2016 = df_clean[df_clean['release year'] == 2016]
df_2017 = df_clean[df_clean['release year'] == 2017]
df_for_yearrating = df_clean[df_clean['release year'].isin([2016, 2017])]

fig2 = px.box(
    df_for_yearrating,
    x="release year",
    y="user rating score",
    color="release year",
    title='Распределение рейтингов по годам выпуска (2016-2017)'
)

fig2.update_layout(
    xaxis_title="Год выпуска",
    yaxis_title="Оценки пользователей",
    template="plotly_white",
    showlegend=False
)

fig2.show()


In [ ]:
print("СРАВНЕНИЕ 2016 vs 2017:")
print(f"Количество релизов 2016: {len(df_2016)}")
print(f"Количество релизов 2017: {len(df_2017)}")
print(f"Средняя оценка 2016: {df_2016['user rating score'].mean():.1f}")
print(f"Средняя оценка 2017: {df_2017['user rating score'].mean():.1f}")

## Распределение рейтингов по жанру

In [ ]:
fig = px.box(
    df_clean,
    x="era",
    y="user rating score",
    color="era",
    title="Распределение рейтингов по эпохам",
    category_orders={"era": sorted(df_clean['era'].unique())}
)

fig.update_layout(
    xaxis_title="Эпоха",
    yaxis_title="Оценка пользователей",
    template="plotly_white",
    showlegend=False
)

fig.show()



## Распределение по возрастным категориям

In [ ]:
fig1 = px.pie(
    df_clean,
    names='age_category',
    title='Распределение контента по возрастным категориям',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig1.update_traces(textposition='inside', textinfo='percent+label')
fig1.show()

## Рейтинг по годам и категориям

In [ ]:
pivot_data = df_clean.pivot_table(
    values='user rating score',
    index='age_category',
    columns='era',
    aggfunc='mean'
)

fig4 = px.imshow(
    pivot_data,
    title='Средний рейтинг: Возрастная категория vs Эпоха',
    labels=dict(x='Эпоха', y='Возрастная категория', color='Рейтинг'),
    color_continuous_scale='RdYlGn'
)
fig4.show()

In [ ]:
rating_stats = df_clean.groupby("age_category").agg({
    "title": "count",
    "user rating score": "mean"
}).reset_index()

rating_stats.rename(columns={"title": "movie_count"}, inplace=True)
rating_stats = rating_stats.sort_values("age_category", ascending=False)

melted_stats = rating_stats.melt(
    id_vars="age_category",
    value_vars=["movie_count", "user rating score"],
    var_name="metric",
    value_name="value"
)

melted_stats["metric"] = melted_stats["metric"].replace({
    "movie_count": "Количество фильмов",
    "user rating score": "Средний рейтинг"
})

fig = px.bar(
    melted_stats,
    x="age_category",
    y="value",
    color="metric",
    barmode="group",
    color_discrete_map={
        "Количество фильмов": "grey",
        "Средний рейтинг": "red"
    },
    title="Сравнение рейтинговых групп по количеству контента и среднему рейтингу",
    labels={
        "age_category": "Возрастной рейтинг",
        "value": "Значение",
        "metric": "Показатель"
    },
    height=600
)

fig.update_layout(
    xaxis=dict(tickangle=-45),
    yaxis=dict(title="Значение"),
    showlegend=True,
)

fig.show()
print(rating_stats)

# **8. Обогащение**

## Расширение данных для более глубокого анализа и подтверждения гипотез



* Netflix Engagement Report (July to December 2023) — показатели просмотров сериалов и фильмов Netflix (столбец `Views`).
* Netflix Movies and TV shows till 2025 — расширенный каталог фильмов и сериалов Netflix с метаданными `country`, `rating`.

Формат обогащения:

* сопоставление по упрощённым названиям `title` (функция `make_title_key` переводит текст в нижний регистр, удаляет символы и обрезает подписи после двоеточия);
* для каждой строки оставляем первое найденное совпадение по просмотрам, стране и рейтингу;
* добавлены столбцы `views`, `country_external`, `rating_external` в очищенный датасет `df_clean`.


### Подготовка данных о просмотрах

In [ ]:
views = pd.read_csv("netflix-engagement-report.csv")

views = views.rename(columns=lambda col: col.strip())

views['Views'] = (
    views['Views']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(' ', '', regex=False)
    .replace({'nan': np.nan, '': np.nan})
)
views['Views'] = pd.to_numeric(views['Views'])


def make_title_key(s):
    if pd.isna(s):
        return np.nan
    text = str(s).lower()
    text = text.split(':')[0]
    text = text.replace('-', ' ').replace("'", ' ')
    text = ''.join(ch if (ch.isalnum() or ch.isspace()) else ' ' for ch in text)
    text = ' '.join(text.split())
    return text


views['title_key'] = views['Title'].map(make_title_key)

views_itog = (
    views
    .dropna(subset=['title_key', 'Views'])
    .sort_values('Views', ascending=False)
    .drop_duplicates(subset='title_key')
    [['title_key', 'Views']]
    .rename(columns={'Views': 'views'})
    .reset_index(drop=True)
)

views_itog = views_itog.set_index('title_key')['views']

df_clean['title_key'] = df_clean['title'].map(make_title_key)
df_clean['views'] = df_clean['title_key'].map(views_itog)
df_clean.drop(columns=['title_key'], inplace=True)

views_itog.head()

### Дополнение странами и рейтингами (Netflix Movies and TV shows till 2025)

### Проверка покрытия и примеры сопоставлений

In [ ]:
movies_2025 = pd.read_csv("netflix_movies_detailed_up_to_2025.csv")
shows_2025 = pd.read_csv("netflix_tv_shows_detailed_up_to_2025.csv")

content_2025 = (
    pd.concat([movies_2025, shows_2025], ignore_index=True, sort=False)
    .rename(columns=lambda col: col.strip())
)

content_2025['title_key'] = content_2025['title'].map(make_title_key)

country_rating_lookup = (
    content_2025
    .dropna(subset=['title_key'])
    [['title_key', 'country', 'rating']]
    .drop_duplicates(subset='title_key')
    .rename(columns={'country': 'country_external', 'rating': 'rating_external'})
    .set_index('title_key')
)


df_clean['title_key'] = df_clean['title'].map(make_title_key)
df_clean = df_clean.merge(
    country_rating_lookup[['country_external', 'rating_external']],
    left_on='title_key',
    right_index=True,
    how='left'
)


### Экспорт обогащённого датасета

In [ ]:
df_clean

# **9. Дополнительные гипотезы + обоснование**

In [ ]:
fig = px.scatter(df_clean,
                 x='views',
                 y='user rating score',
                 color='age_category',
                 title='Зависимость просмотров и рейтинга по годам выпуска',
                 labels={'views': 'Количество просмотров',
                        'user rating score': 'Рейтинг пользователей',
                        'age_category': 'Возрастная категория'},
                 hover_data=['release year'])
fig.update_traces(marker=dict(size=12))
fig.show()

In [ ]:

country_ratings = df_clean.groupby('country_external')['user rating score'].mean().reset_index()

fig = px.choropleth(country_ratings,
                    locations='country_external',
                    locationmode='country names',
                    color='user rating score',
                    color_continuous_scale='inferno',
                    range_color=[country_ratings['user rating score'].min(),
                                country_ratings['user rating score'].max()],
                    title='Средний рейтинг пользователей по странам',
                    labels={'user rating score': 'Средний рейтинг', },
)
fig.update_layout(
    coloraxis_colorbar=dict(title="Рейтинг"),
    width=1000,
    height=600,
)

fig.show()